# Lab 01: Baseline Agent

## Overview

In this notebook, we deploy an **intentionally unoptimized** customer support agent to establish baseline metrics. This baseline will be used to measure improvements in subsequent notebooks.

**What you'll learn:**
- How to deploy an agent to AgentCore Runtime
- How to configure Langfuse for observability
- How to invoke agents via the AgentCore API
- How to establish baseline metrics for cost and latency

## Prerequisites

- AWS account with Bedrock and AgentCore access
- Langfuse account (free tier works)
- `.env` file configured with your credentials

## Workshop Journey

```
[01 Baseline] → 02 Quick Wins → 03 Caching → 04 Routing → 05 Guardrails → 06 Gateway → 07 Evaluations
     ↑
  You are here
```

## Architecture

![Baseline Agent Architecture](images/00_agent_architecture.png)

## Step 1: Setup and Dependencies

Before starting, ensure you've run `uv sync` in the terminal and selected the `.venv` kernel in VS Code.

In [3]:
from __future__ import annotations

import json
import os
import uuid
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(override=True)

import boto3
from bedrock_agentcore_starter_toolkit import Runtime

region = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
control_client = boto3.client("bedrock-agentcore-control", region_name=region)
data_client = boto3.client("bedrock-agentcore", region_name=region)
agentcore_runtime = Runtime()

print(f"Region: {region}")
print(f"Langfuse Host: {os.environ.get('LANGFUSE_BASE_URL', 'Not set')}")

Region: us-east-1
Langfuse Host: https://d1fnzg75c19u2d.cloudfront.net


## Step 2: Review the Baseline Agent

Our baseline agent (`agents/v1_baseline.py`) is intentionally unoptimized:

- **Verbose system prompt** (~1500 tokens instead of ~250)
- **No max_tokens limit** (model can generate unlimited output)
- **No prompt caching** (system prompt processed every request)
- **No stop sequences** (model decides when to stop)

Let's examine the key parts of the baseline agent:

In [5]:
# Read and display the baseline agent code
agent_file = Path("agents/v1_baseline.py")
print(agent_file.read_text())

"""
V1 Baseline Agent - Intentionally unoptimized for comparison.
No caching, verbose prompt, no max_tokens limit.
"""

from __future__ import annotations

import os

from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent
from strands.models import BedrockModel
from strands.telemetry import StrandsTelemetry

from utils.agent_config import MODEL_SONNET, setup_langfuse_telemetry
from utils.tools import get_product_info, get_return_policy, get_technical_support, web_search

setup_langfuse_telemetry()

app = BedrockAgentCoreApp()

# Verbose system prompt - intentionally unoptimized with common prompting mistakes:
# - Dense paragraphs without structure or hierarchy
# - Hedging language ("try to", "hopefully", "if possible", "maybe")
# - Filler phrases ("Please", "Can you please", "It would be great if")
# - No output format specification
# - Redundant adjective chains
# - No few-shot examples
SYSTEM_PROMPT = """
You are a customer support assistant and your job 

## Step 3: Configure the Agent for Deployment

Now we'll configure the agent for deployment to AgentCore Runtime.

In [6]:
agent_name = "customer_support_v1_baseline"
agent_file = str(Path("agents/v1_baseline.py").absolute())
requirements_file = str(Path("requirements-for-agentcore.txt").absolute())

print(f"Agent name: {agent_name}")
print(f"Agent file: {agent_file}")
print(f"Requirements: {requirements_file}")

Agent name: customer_support_v1_baseline
Agent file: /Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/02-developer-journey/agents/v1_baseline.py
Requirements: /Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/02-developer-journey/requirements-for-agentcore.txt


In [7]:
# Clean up any existing build files from previous labs
for f in ["Dockerfile", ".dockerignore", ".bedrock_agentcore.yaml"]:
    p = Path(f)
    if p.exists():
        p.unlink()
        print(f"Removed existing: {f}")

# Configure the runtime
print(f"Configuring agent: {agent_name}")
agentcore_runtime.configure(
    entrypoint=agent_file,
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file=requirements_file,
    region=region,
    agent_name=agent_name,
)

Entrypoint parsed: file=/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/02-developer-journey/agents/v1_baseline.py, bedrock_agentcore_name=v1_baseline
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: customer_support_v1_baseline


Removed existing: Dockerfile
Removed existing: .dockerignore
Removed existing: .bedrock_agentcore.yaml
Configuring agent: customer_support_v1_baseline


💡 No container engine found (Docker/Finch/Podman not installed)

✓ Default deployment uses CodeBuild (no container engine needed), For local builds, install Docker, Finch, or 
Podman

Memory disabled
Network mode: PUBLIC


📄 Generated Dockerfile: 
/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/02-developer-journey/Dockerfile

Generated .dockerignore: /Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/02-developer-journey/.dockerignore
Setting 'customer_support_v1_baseline' as default agent
Bedrock AgentCore configured: /Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/02-developer-journey/.bedrock_agentcore.yaml


ConfigureResult(config_path=PosixPath('/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/02-developer-journey/.bedrock_agentcore.yaml'), dockerfile_path=PosixPath('/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/02-developer-journey/Dockerfile'), dockerignore_path=PosixPath('/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/02-developer-journey/.dockerignore'), runtime='None', runtime_type=None, region='us-east-1', account_id='739907928487', execution_role=None, ecr_repository=None, auto_create_ecr=True, s3_path=None, auto_create_s3=False, memory_id=None, network_mode='PUBLIC', network_subnets=None, network_security_groups=None, network_vpc_id=None)

## Step 4: Modify Dockerfile for Langfuse

We need to disable the default OpenTelemetry instrumentation and use Langfuse instead.

In [8]:
dockerfile_path = Path("Dockerfile")
if dockerfile_path.exists():
    content = dockerfile_path.read_text()
    # Replace opentelemetry-instrument wrapper with direct python call
    # Keep the correct module path: agents.v1_baseline
    if "opentelemetry-instrument" in content:
        import re

        content = re.sub(
            r'CMD \["opentelemetry-instrument", "python", "-m", "([^"]+)"\]', r'CMD ["python", "-m", "\1"]', content
        )
        dockerfile_path.write_text(content)
        print("Dockerfile modified for Langfuse")
    else:
        print("Dockerfile already configured or using different format")
else:
    print("Dockerfile not found - will be created during deployment")

Dockerfile modified for Langfuse


## Step 5: Deploy to AgentCore Runtime

Deploy the agent with Langfuse environment variables.

In [ ]:
env_vars = {
    "LANGFUSE_BASE_URL": os.environ.get("LANGFUSE_BASE_URL"),
    "LANGFUSE_PUBLIC_KEY": os.environ.get("LANGFUSE_PUBLIC_KEY"),
    "LANGFUSE_SECRET_KEY": os.environ.get("LANGFUSE_SECRET_KEY"),
    "PYTHONUNBUFFERED": "1",
}

print("Deploying to AgentCore Runtime...")
print("This may take 5-10 minutes for the first deployment.")
launch_result = agentcore_runtime.launch(env_vars=env_vars, auto_update_on_conflict=True)
print(f"Agent deployed: {launch_result.agent_arn}")

🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'customer_support_v1_baseline' to account 739907928487 (us-east-1)
Generated image tag: 20260601-063122-387
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: customer_support_v1_baseline


Deploying to AgentCore Runtime...
This may take 5-10 minutes for the first deployment.


ECR repository available: 739907928487.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-customer_support_v1_baseline
Getting or creating execution role for agent: customer_support_v1_baseline
Using AWS region: us-east-1, account ID: 739907928487
Role name: AmazonBedrockAgentCoreSDKRuntime-us-east-1-d683549b2a


✅ Reusing existing ECR repository: 739907928487.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-customer_support_v1_baseline


✅ Reusing existing execution role: arn:aws:iam::739907928487:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-d683549b2a
Execution role available: arn:aws:iam::739907928487:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-d683549b2a
Preparing CodeBuild project and uploading source...


In [ ]:
# Save the agent ARN for later use
agent_arn = launch_result.agent_arn
print(f"Agent ARN: {agent_arn}")

# Grant the auto-created execution role the shared runtime permissions
# (SSM KB lookup + Bedrock Retrieve + ApplyGuardrail) used across all labs.
from utils.runtime_helpers import ensure_runtime_permissions

ensure_runtime_permissions(region)

## Step 6: Test the Baseline Agent

Let's invoke the agent with some test queries to establish our baseline metrics.

In [ ]:
def invoke_agent(prompt):
    """Invoke the agent via AgentCore API."""
    response = data_client.invoke_agent_runtime(
        agentRuntimeArn=agent_arn,
        runtimeSessionId=str(uuid.uuid4()),
        payload=json.dumps({"prompt": prompt}).encode(),
    )
    return json.loads(response["response"].read().decode("utf-8"))

In [ ]:
# Import Langfuse metrics helper
from utils.langfuse_metrics import (
    clear_metrics,
    collect_metric,
    get_latest_trace_metrics,
    print_metrics,
    print_metrics_table,
)

# Clear any previously collected metrics
clear_metrics()


def invoke_agent_with_metrics(prompt, test_name=""):
    """Invoke the agent and fetch + print Langfuse metrics."""
    response = invoke_agent(prompt)
    print(response)

    # Fetch and print metrics from Langfuse
    metrics = get_latest_trace_metrics(
        agent_name="customer-support-v1-baseline",
        wait_seconds=5,
        max_retries=5,
        timeout_seconds=120,
    )
    print_metrics(metrics, test_name)

    # Collect metrics for summary table
    collect_metric(metrics, test_name)

    return response, metrics

In [ ]:
# Standard test prompts - same across all notebooks for consistent comparison
TEST_PROMPTS = [
    # Single tool: get_return_policy
    ("Return Policy", "What is your return policy for laptops?"),
    # Single tool: get_product_info
    ("Product Info", "Tell me about your smartphone options"),
    # Single tool: get_technical_support (Bedrock KB)
    ("Technical Support", "My laptop won't turn on, can you help me troubleshoot?"),
    # Multi-tool: get_product_info + get_return_policy
    ("Multi-part Question", "I want to buy a laptop. What are the specs and what's the return policy?"),
    # No tool: General greeting
    ("General Question", "Hello! What can you help me with today?"),
]

# Run all tests and collect metrics
for test_name, prompt in TEST_PROMPTS:
    print("=" * 60)
    print(f"Test: {test_name}")
    print("=" * 60)
    response, metrics = invoke_agent_with_metrics(prompt, test_name=test_name)

Test: Return Policy
Great news — here's a full breakdown of our return policy for laptops at TechMart Electronics:

**📦 Return Window**
You have **30 days** from the date of purchase to return a laptop.

**✅ Condition Requirements**
- Must be in **original packaging**
- All **accessories and components** must be included
- No **physical damage**

**🔄 Return Process**
You can initiate a return through our **online RMA portal** or by visiting us **in-store**.

**💰 Refund Timeline**
Once your return is received and inspected, you can expect your refund within **7–10 business days**.

**🚚 Return Shipping**
- **Free return shipping** if the laptop has a defect
- **Customer pays shipping** for change-of-mind returns

**💸 Restocking Fee**
- **No restocking fee** for defective items
- **15% restocking fee** for change-of-mind returns

**🛡️ Warranty**
Laptops come with a **1-year manufacturer warranty**, and extended warranty options are also available if you'd like additional coverage.

Is the

In [ ]:
# Print summary table of all test metrics
print_metrics_table()

# Save metrics for comparison in later notebooks
from utils.langfuse_metrics import save_metrics
save_metrics("v1")


                                  METRICS SUMMARY
               Test Latency Cost Input Output Cache Read Tokens Cache Write Tokens
      Return Policy   ERROR    -     -      -                 -                  -
       Product Info   ERROR    -     -      -                 -                  -
  Technical Support   ERROR    -     -      -                 -                  -
Multi-part Question   ERROR    -     -      -                 -                  -
   General Question   ERROR    -     -      -                 -                  -
---------------------------------------------------------------------------------------------------------
  TOTALS: Latency(avg): 0.00s | Cost: $0.0000 | Input: 0 | Output: 0
          Cache Read Tokens: 0 | Cache Write Tokens: 0

Metrics saved as 'v1' → .lab_metrics.json


## Step 7: View Langfuse Dashboard

Now let's check the Langfuse dashboard to see our baseline metrics.

### What to look for:

1. **Token Usage**: Note the input and output token counts
2. **Latency**: Time taken for each request
3. **Cost**: Estimated cost per request
4. **No Cache Hits**: `cacheReadInputTokens` should be 0 (no caching)

### Dashboard URL

In [ ]:
langfuse_base_url = os.environ.get("LANGFUSE_BASE_URL", "https://cloud.langfuse.com")
print(f"View your traces at: {langfuse_base_url}")
print("\nFilter by tags: 'baseline', 'no-optimization'")
print("\nMetrics to record:")
print("- Average input tokens: _____")
print("- Average output tokens: _____")
print("- Average latency: _____ ms")
print("- Cache read tokens: 0 (expected)")

View your traces at: http://localhost:3000

Filter by tags: 'baseline', 'no-optimization'

Metrics to record:
- Average input tokens: _____
- Average output tokens: _____
- Average latency: _____ ms
- Cache read tokens: 0 (expected)


## Summary

In this notebook, we:

1. Deployed an unoptimized baseline agent to AgentCore Runtime
2. Configured Langfuse for observability
3. Ran test scenarios to establish baseline metrics
4. Identified areas for optimization:
   - Verbose system prompt (~1500 tokens)
   - No output token limits
   - No prompt caching
   - No model routing

**Next Steps:** In the next notebook, we'll apply "quick wins" optimizations:
- Concise system prompt
- max_tokens limit
- stop_sequences

**Next notebook:** [02-quick-wins.ipynb](./02-quick-wins.ipynb)

## Cleanup (Optional)

Uncomment and run if you want to delete the agent.

In [ ]:
# # Uncomment to delete resources created in this lab
# agentcore_runtime.destroy(delete_ecr_repo=True)
# print(f"Deleted agent and ECR repository: {agent_name}")